<a href="https://colab.research.google.com/github/david-levin11/Verification_Notebooks/blob/main/NBM_Version_Compare.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**NBM Experimental and Operational Archive Viewer**
This notebook will plot various probabilistic fields from the NBM archive on the NODD as as 6 panel plot.  It will also give you a difference plot between the operational (currently v4.3) and the experimental (currently v5.0) where both fields exist on the NODD.

- David Levin (david.levin@noaa.gov) Arctic Testbed & Proving Ground adapted loosely from Steve Fleegel's NBM 6-Panel notebook.

**Data Availability:** <br>
NBM 4.x: 5/18/2020 to Present <br>
NBM 5.0 (experimental): 4 day running archive from today

## **Run this to download, install, and import the appropriate packages.**  
This first cell will take around a minute to run and get everything ready for us to download the data and make the image(s).

In [1]:
# @title
# Install the packages needed for this notebook.
!pip install -q eccodes==2.38.3 --progress-bar off ## Fixes issue with ecCodes 2.39.0 and Herbie causing Google Colab to crash
!pip install -q cartopy contextily pyepsg xesmf netCDF4 eccodes --progress-bar off
!pip install -q pygrib

## Imports
import pandas as pd
import pygrib
from datetime import datetime, timedelta
import requests
import re
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
import matplotlib.colors as mcolors
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch
import cartopy.crs as ccrs
import cartopy.feature as cfeature
# Downlaod and Import shapefiles
import cartopy.io.shapereader as shpreader
import os
import geopandas as gpd
from shapely.geometry import box
import numpy as np
import xarray as xr

# Set Variables that shouldn't be reset with the script
marineZ_dom = None
marineZ_map_extent = [0,0,0,0]

## Make Folders to store data
!mkdir /content/shp

print ("Install & Import Done!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.6/18.6 MB 93.1 MB/s eta 0:00:00
Install & Import Done!


##**Download Data, prep the data, and create the image**

Make your selections for the model, data, time, etc for the image. When you're ready, click the play button to make your image. The image will show up below and can also be downloaded from the folder to the left.

The first run will take the longest, as it needs to download map data for the background.

In [6]:


# Base Variables
IncludeCounties = False
IncludeCities = False
IncludeMarineZones = False

Map_Fig_Size = (17, 13)
city_removal = []
local_cities_dict = {}

############################ Methods ##########################################


###################### Color Maps ###########################################
def make_custom_cmaps(name, colors, bounds: list = None, N: int = None):
    if N is None:
        N = len(colors)
    linear_cmap = mcolors.LinearSegmentedColormap.from_list(name, colors)
    segment_cmap = mcolors.LinearSegmentedColormap.from_list(name + "2", colors, N=N)

    # When data is NaN, set color to transparent
    linear_cmap.set_bad("#ffffff00")
    segment_cmap.set_bad("#ffffff00")

    for cm in [linear_cmap, segment_cmap]:
        mpl.colormaps.register(cmap=cm, force=True)
        mpl.colormaps.register(cmap=cm.reversed(), force=True)

    if bounds is not None:
        return (
            mcolors.Normalize(bounds.min(), bounds.max()),
            mcolors.BoundaryNorm(bounds, linear_cmap.N),
        )

class NWSPrecipitation:
    """National Weather Service precipitation amount colorbar properties.

    Also known as Qualitative Precipitation Forecast/Estimate (QPF/QPE).
    """

    name = "nws.pcp"
    units = "in"
    variable = "Precipitation"
    colors = np.array(
        [
            "#ffffff",
            "#c7e9c0",
            "#a1d99b",
            "#74c476",
            "#31a353",
            "#006d2c",
            "#fffa8a",
            "#ffcc4f",
            "#fe8d3c",
            "#fc4e2a",
            "#d61a1c",
            "#ad0026",
            "#700026",
            "#3b0030",
            "#4c0073",
            "#ffdbff",
        ]
    )
    # NWS bounds in inches
    bounds = np.array(
        [0, 0.01, 0.1, 0.25, 0.5, 1, 1.5, 2, 3, 4, 6, 8, 10, 15, 20, 30, 50]
    )
    norm, norm2 = make_custom_cmaps(name, colors, bounds)
    cmap = plt.get_cmap(name)
    cmap2 = plt.get_cmap(name + "2")
    kwargs = dict(cmap=cmap, norm=norm)
    kwargs2 = dict(cmap=cmap, norm=norm2)
    cbar_kwargs = dict(label=f"{variable} ({units})")
    cbar_kwargs2 = cbar_kwargs | dict(spacing="uniform", ticks=bounds)

class NWSPrecipitationSnow:
    name = "nws.pcp_snow"
    units = "in"
    variable = "Snow"
    colors = np.array(
        [
            "#ffffff",
            "#bdd7e7",
            "#6baed6",
            "#3182bd",
            "#08519c",
            "#082694",
            "#ffff96",
            "#ffc400",
            "#ff8700",
            "#db1400",
            "#9e0000",
            "#690000",
            "#360000",
        ]
    )
    # NWS bounds in inches
    bounds = np.array([0, 0.1, 1, 2, 3, 4, 6, 8, 12, 18, 24, 30, 36, 42])
    norm, norm2 = make_custom_cmaps(name, colors, bounds)
    cmap = plt.get_cmap(name)
    cmap2 = plt.get_cmap(name + "2")
    kwargs = dict(cmap=cmap, norm=norm)
    kwargs2 = dict(cmap=cmap, norm=norm2)
    cbar_kwargs = dict(label=f"{variable} ({units})")
    cbar_kwargs2 = cbar_kwargs | dict(spacing="uniform", ticks=bounds)

class NWSWindSpeed:
    name = "nws.wind"
    units = r"mph"
    variable = "Wind Speed"
    colors = np.array(
        [
            "#103f78",
            "#225ea8",
            "#1d91c0",
            "#41b6c4",
            "#7fcdbb",
            "#b4d79e",
            "#dfff9e",
            "#ffffa6",
            "#ffe873",
            "#ffc400",
            "#ffaa00",
            "#ff5900",
            "#ff0000",
            "#a80000",
            "#6e0000",
            "#ffbee8",
            "#ff73df",
        ]
    )
    # MPH
    bounds = np.array(
        [0.0, 5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 60, 70, 80, 100, 120, 140, 160]
    )
    norm, norm2 = make_custom_cmaps(name, colors, bounds)
    cmap = plt.get_cmap(name)
    cmap2 = plt.get_cmap(name + "2")
    kwargs = dict(cmap=cmap, norm=norm)
    kwargs2 = dict(cmap=cmap, norm=norm2)
    cbar_kwargs = dict(label=f"{variable} ({units})")
    cbar_kwargs2 = cbar_kwargs | dict(spacing="proportional", ticks=bounds)

class NWSWindSpeedkts:
    name = "nws.wind"
    units = r"kts"
    variable = "Wind Speed"
    colors = np.array(
        [
            "#103f78",
            "#225ea8",
            "#1d91c0",
            "#41b6c4",
            "#7fcdbb",
            "#b4d79e",
            "#dfff9e",
            "#ffffa6",
            "#ffe873",
            "#ffc400",
            "#ffaa00",
            "#ff5900",
            "#ff0000",
            "#a80000",
            "#6e0000",
            "#ffbee8",
            "#ff73df",
        ]
    )
    # kts
    bounds = np.array(
        [0.0, 5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 60, 70, 80, 100, 120, 140, 160]
    )
    norm, norm2 = make_custom_cmaps(name, colors, bounds)
    cmap = plt.get_cmap(name)
    cmap2 = plt.get_cmap(name + "2")
    kwargs = dict(cmap=cmap, norm=norm)
    kwargs2 = dict(cmap=cmap, norm=norm2)
    cbar_kwargs = dict(label=f"{variable} ({units})")
    cbar_kwargs2 = cbar_kwargs | dict(spacing="proportional", ticks=bounds)

############################ Resuable Methods ###############################
#Building search strings
def build_search_string(forecast_hour, percentile, percentile_duration, element):
    if element == "Snow-Pctl" or element == "QPF-Pctl":
        if percentile_duration == '72hr':
          search_hr_range = str(forecast_hour-72) + "-" + str(forecast_hour)
        elif percentile_duration == "48hr":
          search_hr_range = str(forecast_hour-48) + "-" + str(forecast_hour)
        elif percentile_duration == "24hr":
          search_hr_range = str(forecast_hour-24) + "-" + str(forecast_hour)
    else:
        search_hr_range = str(forecast_hour)

    if element == "Snow-Pctl" and percentile == "Det":
        SearchParam = ":ASNOW:surface:" + str(search_hr_range) + " hour acc fcst(?::)?$"
    elif element == "Snow-Pctl" and percentile != "Det":
        SearchParam = ":ASNOW:surface:" + str(search_hr_range) + " hour acc fcst:" + percentile +" level"

    elif element == "Gust-Pctl" and percentile == "Det":
        SearchParam = ":GUST:10 m above ground:" + str(search_hr_range) + " hour fcst"
    elif elements == "Gust-Pctl" and percentile != "Det":
        SearchParam = ":GUST:10 m above ground:" + str(search_hr_range) + " hour fcst:" + percentile +" level"

    elif element == "Wind-Pctl" and percentile == "Det":
        SearchParam = ":WIND:10 m above ground:" + str(search_hr_range) + " hour fcst:"
    elif element == "Wind-Pctl" and percentile != "Det":
        SearchParam = ":WIND:10 m above ground:" + str(search_hr_range) + " hour fcst:" + percentile +" level"

    elif element == "QPF-Pctl" and percentile == "Det":
        SearchParam = ":APCP:surface:" + str(search_hr_range) + " hour acc fcst(?::)?$"
    elif element == "QPF-Pctl" and percentile != "Det":
        SearchParam = ":APCP:surface:" + str(search_hr_range) + " hour acc fcst:" + percentile +" level"
    return SearchParam

def mm_to_in(mm):
    return mm * 0.0393701

def m_to_in(m):
    return m * 39.3701

def ms_to_kts(ms):
    return ms * 1.94384

def ms_to_mph(ms):
    return ms * 2.23694

def download_subset_dev(remote_url, remote_file, local_filename, search_string, percentile=True):
    print("   > Downloading a subset of NBM gribs")
    local_file = os.path.join("nbm", local_filename)
    os.makedirs(os.path.dirname(local_file), exist_ok=True)

    # Download the .idx file
    idx_url = remote_url + ".idx"
    r = requests.get(idx_url)
    if not r.ok:
        print(f'     ❌ Failed to fetch index file: {idx_url} ({r.status_code} {r.reason})')
        return None

    lines = r.text.strip().split('\n')

    expr = re.compile(search_string)

    byte_ranges = {}
    if percentile:
      for n, line in enumerate(lines):
          if expr.search(line):
              parts = line.split(':')
              rangestart = int(parts[1])
              if n + 1 < len(lines):
                  next_parts = lines[n + 1].split(':')
                  rangeend = int(next_parts[1]) - 1
              else:
                  rangeend = ''

              byte_range = f'{rangestart}-{rangeend}' if rangeend else f'{rangestart}-'
              byte_ranges[byte_range] = line
    else:
      for n, line in enumerate(lines):
          if expr.search(line) and "% level" not in line and "Probability of event" not in line:
              parts = line.split(':')
              rangestart = int(parts[1])
              if n + 1 < len(lines):
                  next_parts = lines[n + 1].split(':')
                  rangeend = int(next_parts[1]) - 1
              else:
                  rangeend = ''

              byte_range = f'{rangestart}-{rangeend}' if rangeend else f'{rangestart}-'
              byte_ranges[byte_range] = line

    if not byte_ranges:
        print(f'      ❌ No matches found for [{search_string}]')
        return None

    for i, (byte_range, line) in enumerate(byte_ranges.items()):
        headers = {'Range': f'bytes={byte_range}'}
        r = requests.get(remote_url, headers=headers)
        if r.status_code not in (200, 206):
            print(f"      ❌ Failed to download byte range {byte_range}")
            return None
        with open(local_file, 'ab' if i > 0 else 'wb') as f:
            f.write(r.content)

    print(f'      ✅ Downloaded [{len(byte_ranges)}] fields → {local_file}')
    return local_file

def plot_towns(ax, south, north, west, east, population=1000, resolution='10m', transform=ccrs.PlateCarree(), zorder=3):
    """
    This function will download the 'populated_places' shapefile from
    NaturalEarth, trim the shapefile based on the limits of the provided
    lat & long coords, and then plot the locations and names of the towns
    on a given GeoAxes.

    ax = a pyplot axes object
    south = south lat limit (float)#
    north = north lat limit (float)
    west = west long limit (float)
    east = east long limit (float)
    resolution= str. either high res:'10m' or low res: '50m'
    population = minimum population of towns to plot (int)
    transform = a cartopy crs object
    """
    #get town locations
    shp_fn = shpreader.natural_earth(resolution=resolution, category='cultural', name='populated_places')
    shp = shpreader.Reader(shp_fn)
    xy = [pt.coords[0] for pt in shp.geometries()]
    x, y = list(zip(*xy))

    #get town names
    towns = shp.records()
    names_en = []
    max_population = []
    for town in towns:
        #print(town.attributes)
        names = town.attributes['NAME']
        pop = town.attributes['POP_MAX']
        names_en.append(names)
        max_population.append(pop)
    #print(names_en)
    #create data frame and index by the region of the plot
    all_towns = pd.DataFrame({'names_en': names_en, 'x':x, 'y':y, 'population':max_population})
    #print(all_towns.head())
    region_towns = all_towns[(all_towns.y<north) & (all_towns.y>south)
                           & (all_towns.x>west) & (all_towns.x<east)]
    region_towns = region_towns[region_towns.population > population]
    #print(region_towns.head())
    #plot the locations and labels of the towns in the region
    ax.scatter(region_towns.x.values, region_towns.y.values, c ='black', marker= '.', transform=transform, zorder=zorder)
    transform_mpl = ccrs.PlateCarree()._as_mpl_transform(ax) #this is a work-around to transform xy coords in ax.annotate
    for i, txt in enumerate(region_towns.names_en):
         ax.annotate(txt, (region_towns.x.values[i], region_towns.y.values[i]), xycoords=transform_mpl)

def plot_six_panel_grib2(element, filenames, titles, domain_extent, projection, cmap, cbar, wind_units, difference_mode=False, diffdata=[]):
    fig, axes = plt.subplots(3, 2, figsize=Map_Fig_Size, subplot_kw={'projection': projection})
    axes = axes.flatten()
    if not difference_mode:
      for i, (ax, file) in enumerate(zip(axes, filenames)):
          if not file:
              ax.set_title("Missing Data")
              ax.coastlines()
              continue

          try:
              grbs = pygrib.open(f"./nbm/{file}")
              g = grbs[1]  # Assuming subset file has one variable
              if element == "QPF-Pctl":
                data = g.values
                data = mm_to_in(data)
              elif element == "Wind-Pctl" and wind_units == "kts":
                data = g.values
                data = ms_to_kts(data)
              elif element == "Wind-Pctl" and wind_units == "mph":
                data = g.values
                data = ms_to_mph(data)
              elif element == "Gust-Pctl" and wind_units == "kts":
                data = g.values
                data = ms_to_kts(data)
              elif element == "Gust-Pctl" and wind_units == "mph":
                data = g.values
                data = ms_to_mph(data)
              else:
                raise NotImplementedError(f"Plotting not set up for {element}")
              lats, lons = g.latlons()
              title = titles[i]
              ax.set_extent(domain_extent, crs=ccrs.PlateCarree())
              # Add features to the map
              ax.coastlines(resolution='10m', linewidth=1)
              ax.add_feature(cfeature.BORDERS, linestyle=':')
              ax.add_feature(cfeature.STATES, edgecolor='gray', linestyle='--')
              plot_towns(ax, domain_extent[2], domain_extent[3], domain_extent[0], domain_extent[1], population=500, resolution='10m', transform=ccrs.PlateCarree(), zorder=3)
              cs = ax.pcolormesh(lons, lats, data, transform=ccrs.PlateCarree(), **cmap)
              ax.set_title(title)
              fig.colorbar(cs, ax=ax, orientation='vertical', pad=0.01, aspect=20, **cbar)
          except Exception as e:
              ax.set_title(f"Error: {e}")
      modelname = titles[0].split(" ")[0]+titles[0].split(" ")[1]
    else:
      colorbar_scale = {
        "QPF-Pctl": [-1.5, 1.5],
        "Wind-Pctl": [-10.0, 10.0],
        "Gust-Pctl": [-15.0, 15.0]
      }
      for i, (ax, file, values) in enumerate(zip(axes, filenames, diffdata)):
          try:
              grbs = pygrib.open(f"./nbm/{file}")
              g = grbs[1]  # Assuming subset file has one variable
              if element == "QPF-Pctl":
                data = mm_to_in(values)
              elif element == "Wind-Pctl" and wind_units == "kts":
                data = ms_to_kts(values)
              elif element == "Wind-Pctl" and wind_units == "mph":
                data = ms_to_mph(values)
              elif element == "Gust-Pctl" and wind_units == "kts":
                data = ms_to_kts(values)
              elif element == "Gust-Pctl" and wind_units == "mph":
                data = ms_to_mph(values)
              else:
                raise NotImplementedError(f"Plotting not set up for {element}")
              #data = values
              #print(f"Data is: {data}")
              #print(f"Shape is: {data.shape}")
              lats, lons = g.latlons()
              title = titles[i]
              ax.set_extent(domain_extent, crs=ccrs.PlateCarree())
              ax.coastlines()
              ax.add_feature(cfeature.BORDERS, linestyle=':')
              ax.add_feature(cfeature.STATES, edgecolor='gray', linestyle='--')
              plot_towns(ax, domain_extent[2], domain_extent[3], domain_extent[0], domain_extent[1], population=500, resolution='10m', transform=ccrs.PlateCarree(), zorder=3)
              if element == "QPF-Pctl":
                cs = ax.pcolormesh(lons, lats, data, transform=ccrs.PlateCarree(), cmap='BrBG',vmin=colorbar_scale[element][0], vmax=colorbar_scale[element][1])
              elif element == "Wind-Pctl":
                cs = ax.pcolormesh(lons, lats, data, transform=ccrs.PlateCarree(), cmap='seismic',vmin=colorbar_scale[element][0], vmax=colorbar_scale[element][1])
              ax.set_title(title)
              fig.colorbar(cs, ax=ax, orientation='vertical', pad=0.01, aspect=20)
          except Exception as e:
              ax.set_title(f"Error: {e}")
              ax.coastlines()
      modelname = titles[0].split(" ")[0]+titles[0].split(" ")[1]+"_"+titles[0].split(" ")[2]+titles[0].split(" ")[3]
    plt.tight_layout()
    graphic_title = f"{modelname}_{element}_{nbm_init_date}_{valid_date}_{valid_time}.png"
    os.makedirs("/content", exist_ok=True)
    plt.savefig(os.path.join(f"/content/{graphic_title}"))
    plt.close()
    print(f"Saved {graphic_title} to /content/{graphic_title}")


####################### Config and Markdown ###################################

#@markdown # **Initial Forecast Info**
#@markdown <b>Select Model Run:</b><br />
nbm_init_date = "2025-07-31" #@param {type:"date"}
nbm_init_hour = 0 #@param {type:"slider", min:0, max:23, step:6}
#@markdown ###### If making a ProbSnow image, make sure to choose a 01Z/13Z model run and make sure the forecasts time ends at 00Z/06Z/12Z/18Z
Model = "NBM v5.0" #@param ['NBM v4.3', 'NBM v5.0']
#@markdown Would you like to compare versions? Check the box below
version_compare = True #@param {type:"boolean"}
##@markdown <b>Select forecast hour:</b><br />
#fhr = 21 #@param {type:"slider", min:0, max:257, step:1}
#step = np.arange(0, 49, 1, dtype=int)
#dt = str.split(Model_Date,'-')

#@markdown <b>Select Forecast Valid Time</b><br />
valid_date = "2025-08-01" #@param {type:"date"}
valid_time = 12 #@param {type:"slider", min:0, max:23, step:6}

#@markdown <b>What Percenitle Data do you want to display?</b><br />
#@markdown Wind and Gust percentiles are only available for Alaska for > NBM 5.0
elements = "QPF-Pctl" #@param ['Snow-Pctl', 'Wind-Pctl', 'Gust-Pctl', 'QPF-Pctl']
SnowQPFPctl_Duration = '24hr'  #@param ['72hr', '48hr', '24hr']
percentile_length = int(SnowQPFPctl_Duration.replace('hr', ''))
#@markdown #### Select which percentile for each section of the plot (UL = Upper left etc)
Pctl_UL = "10%" #@param ['5%', '10%', '25%', '50%','75%', '90%', '95%', 'Det']
Pctl_UM = "25%" #@param ['5%', '10%', '25%', '50%','75%', '90%', '95%', 'Det']
Pctl_UR = "Det" #@param ['5%', '10%', '25%', '50%','75%', '90%', '95%', 'Det']
Pctl_LL = "50%" #@param ['5%', '10%', '25%', '50%','75%', '90%', '95%', 'Det']
Pctl_LM = "75%" #@param ['5%', '10%', '25%', '50%','75%', '90%', '95%', 'Det']
Pctl_LR = "90%" #@param ['5%', '10%', '25%', '50%','75%', '90%', '95%', 'Det']

#@markdown # **Map: Area and Type**
#@markdown ### Select Map Info - Zoom:
Map_Zoom = "Anchorage Area" #@param ["AR", "AJK", "AFC", "AFG", "Fairbanks Area", "Anchorage Area", "Juneau Area", "Custom"]

#@markdown ### If a Custom domain is selected, adjust the items below<br>
#@markdown Enter a custom name if you want (mainly used for filename)
custom_name = "Fairbanks" #@param {type:"string"}
#@markdown Enter custom lat/lon bounding box if custom was selected
custom_bottom_lat =  63.35 #@param {type:"number"}
custom_left_lon = -153 #@param {type:"number"}
custom_top_lat = 66 #@param {type:"number"}
custom_right_lon = -142.75 #@param {type:"number"}

# @markdown ###If needed, adjust for map panel spacing
# @markdown ##### Edit one or more of the following values - may require some testing</i></b></font>
# @markdown <ol><li>Adjust your lat/lon boundary ranges</li>
# @markdown <li>Adjust the height and width dimensions of the figure.</li>
## @markdown <li>Adjust the plot aspect ratio ... >1 shrinks horizontal scale and vice versa.</li></ol>
## @markdown The default values of a height of 13 and a width of 17 work well with a traditional rectangle shape.
fig_height = 25 # @param {type:"raw"}
fig_height = int(fig_height)
fig_width = 25 # @param {type:"raw"}
fig_width = int(fig_width)
##plot_aspect_ratio = 1.25 # @param {type:"raw"}
##plot_aspect_ratio = float(plot_aspect_ratio)


#@markdown # **Individual Map Type Selections**
##@markdown #### <b>Do you want to include cities?</b>
#IncludeCities = False #@param ["True", "False"] {type:"raw"}
#@markdown #### <b>What units do you want the wind in?</b>
Wind_Units = 'mph'  #@param ['knots', 'mph']
#@markdown #### <b> Do you want NWS Legend?</b>
nwslegend_opt = True #@param {type:"boolean"}
#@markdown #### <b> Do you want to plot Marine Zones?</b>
IncludeMarineZones = False #@param {type:"boolean"}

## Define Map Extents

## Adding Alaska check
if Map_Zoom in ["AJK", "AFG", "AFC", "AR", "Anchorage Area", "Juneau Area", "Fairbanks Area"]:
  domain = "ak"
  print(f"Using Map_Zoom list logic...")
elif Map_Zoom == "Custom" and custom_bottom_lat > 50:
  domain = "ak"
  print(f"Using Map_Zoom custom logic...")
else:
  domain = "co"

print(f"Domain is: {domain}")

if Map_Zoom == "AR":
    #State of AK
    Map_Fig_Size = (16, 7)
    Map_Extent = [-179.00, -129.00, 52.00, 72.00]
    IncludeCounties = True
    IncludeMarineZones = True
elif Map_Zoom == "AJK":
    Map_Fig_Size = (12, 15)
    Map_Extent = [-145.00, -129.00, 53.00, 61.00]
    IncludeCounties = True
    IncludeMarineZones = True
elif Map_Zoom == "Juneau Area":
    Map_Fig_Size = (15, 15)
    Map_Extent = [-135.65, -133.53, 57.59, 58.85]
    IncludeCounties = True
    IncludeMarineZones = True
elif Map_Zoom == "AFG":
    Map_Fig_Size = (15, 11)
    Map_Extent = [-170.00, -141.00, 62.00, 72.00]
    IncludeCounties = True
    IncludeMarineZones = True
elif Map_Zoom == "Fairbanks Area":
    Map_Fig_Size = (15, 15)
    Map_Extent = [-150.32, -144.66, 63.69, 65.51]
    IncludeCounties = True
elif Map_Zoom == "AFC":
    Map_Fig_Size = (16, 7)
    Map_Extent = [-160.00, -140.00, 56.00, 63.00]
    IncludeCounties = True
    IncludeMarineZones = True
elif Map_Zoom == "Anchorage Area":
    Map_Fig_Size = (15, 15)
    Map_Extent = [-154.65, -146.94, 59.17, 62.52]
    IncludeCounties = True
    IncludeMarineZones = True
elif Map_Zoom == "Custom":
    # Custom
    Map_Fig_Size = (fig_width, fig_height)
    Map_Extent = [custom_left_lon, custom_right_lon, custom_bottom_lat, custom_top_lat]
    IncludeCounties = True

# Set Projection
# Make the Central Lat/Lon in the middle of the map
cent_lon = ((Map_Extent[0] - Map_Extent[1])/2) + Map_Extent[1]
cent_lat = ((Map_Extent[3] - Map_Extent[2])/2) + Map_Extent[2]
# Assign projection
#crs = ccrs.LambertConformal(central_longitude=-100.0, central_latitude=45.0)
if domain == "ak":
  crs = ccrs.NorthPolarStereo(true_scale_latitude=60.0, central_longitude=225)
elif domain == "co":
  crs = ccrs.LambertConformal(central_longitude=cent_lon, central_latitude=cent_lat)
else:
  raise NotImplementedError(f"Plotting not set up for {domain}")

# setting colorbars
if elements == "QPF-Pctl":
  cmap = NWSPrecipitation.kwargs2
  cbar = NWSPrecipitation.cbar_kwargs2
elif elements == "Wind-Pctl" and Wind_Units == "mph":
  cmap = NWSWindSpeed.kwargs2
  cbar = NWSWindSpeed.cbar_kwargs2
elif elements == "Wind-Pctl" and Wind_Units == "kts":
  cmap = NWSWindSpeedkts.kwargs2
  cbar = NWSWindSpeedkts.cbar_kwargs2
elif elements == "Gust-Pctl" and Wind_Units == "mph":
  cmap = NWSWindSpeed.kwargs2
  cbar = NWSWindSpeed.cbar_kwargs2
elif elements == "Gust-Pctl" and Wind_Units == "kts":
  cmap = NWSWindSpeedkts.kwargs2
  cbar = NWSWindSpeedkts.cbar_kwargs2
elif elements == "Snow-Pctl":
  cmap = NWSPrecipitationSnow.kwargs2
  cbar = NWSPrecipitationSnow.cbar_kwargs2
else:
  raise NotImplementedError(f"Plotting not set up for {elements}")

# setting up our datetimes and checking
nbm_init = datetime.strptime(nbm_init_date,'%Y-%m-%d') + timedelta(hours=int(nbm_init_hour))
core_init = nbm_init
valid_datetime = datetime.strptime(valid_date,'%Y-%m-%d') + timedelta(hours=int(valid_time))
nbm_qmd_forecasthour = int((valid_datetime - nbm_init).total_seconds()/3600)
#print(f"NBM Init Date: {nbm_init}")
#print(f"NBM Valid Date: {valid_datetime}")
#print(f"NBM QMD forecast hour is {nbm_qmd_forecasthour}")

if Model == "NBM v5.0" and nbm_init < datetime.strptime('2025-07-11','%Y-%m-%d'):
  print(f"No exprimental NBM data available prior to 7/11/2025. Please check your date and try again")
  raise SystemExit




if version_compare == True:
  # some version comparing won't be possible for OCONUS
  if domain == "ak" and elements in ["Wind-Pctl", "Gust-Pctl"]:
    print(f"Wind and Gust percentiles are only available for Alaska for > NBM 5.0")
    raise SystemExit
  nbm_url_base = "https://noaa-nbm-grib2-pds.s3.amazonaws.com/blend."+nbm_init.strftime('%Y%m%d') \
              +"/"+nbm_init.strftime('%H')+"/"
  nbm_url_base_core = "https://noaa-nbm-grib2-pds.s3.amazonaws.com/blend."+core_init.strftime('%Y%m%d') \
              +"/"+core_init.strftime('%H')+"/"

  nbm_url_base_exp = "https://noaa-nbm-para-pds.s3.amazonaws.com/blend."+nbm_init.strftime('%Y%m%d') \
              +"/"+nbm_init.strftime('%H')+"/"
  nbm_url_base_core_exp = "https://noaa-nbm-para-pds.s3.amazonaws.com/blend."+core_init.strftime('%Y%m%d') \
              +"/"+core_init.strftime('%H')+"/"

  nbm_init_filen = nbm_init.strftime('%Y%m%d') + "_" + nbm_init.strftime('%H')
  #nbm_init_filen_core = core_init.strftime('%Y%m%d') + "_" + core_init.strftime('%H')
  nbm_init_filen_core = nbm_init_filen
  #print(nbm_url_base)
  #print(nbm_url_base_core)
  # downloading our data
  oper_files = {}
  exp_files = {}
  titles = []
  print(f"Using domain of {domain} to pull data...")
  for field in [Pctl_UL, Pctl_LL, Pctl_UM, Pctl_LM, Pctl_UR, Pctl_LR]:
    if elements in ["QPF-Pctl", "Snow-Pctl"] and field == "Det":
      perc_file = f'blend.t{int(nbm_init_hour):02}z.qmd.f{int(nbm_qmd_forecasthour):03}.{domain}.grib2'
      perc_file_subset = f'blend.t{int(nbm_init_hour):02}z.qmd.{nbm_init_filen}{nbm_init_filen}f{int(nbm_qmd_forecasthour):03}.{domain}.{elements}_Det_subset.grib2'
      perc_url = nbm_url_base+"qmd/"+perc_file
      perc_file_exp = f'blend.t{int(nbm_init_hour):02}z.qmd.f{int(nbm_qmd_forecasthour):03}.{domain}.grib2'
      perc_file_subset_exp = f'blend.t{int(nbm_init_hour):02}z.qmd.{nbm_init_filen}{nbm_init_filen}f{int(nbm_qmd_forecasthour):03}.{domain}.{elements}_Det_subset_exp.grib2'
      perc_url_exp = nbm_url_base_exp+"qmd/"+perc_file
      search_string = build_search_string(int(nbm_qmd_forecasthour), field, SnowQPFPctl_Duration, elements)
      titles.append(f"NBM v5.0 - NBM v4.3 {SnowQPFPctl_Duration} Deterministic {elements.replace('-Pctl','')}")
    elif elements in ["Wind-Pctl", "Gust-Pctl"] and field == "Det":
      perc_file = f'blend.t{int(nbm_init_hour):02}z.core.f{int(nbm_qmd_forecasthour):03}.{domain}.grib2'
      perc_file_subset = f'blend.t{int(nbm_init_hour):02}z.core.{nbm_init_filen}{nbm_init_filen}f{int(nbm_qmd_forecasthour):03}.{domain}.{elements}_Det_subset.grib2'
      perc_url = nbm_url_base+"core/"+perc_file
      perc_file_exp = f'blend.t{int(nbm_init_hour):02}z.core.f{int(nbm_qmd_forecasthour):03}.{domain}.grib2'
      perc_file_subset_exp = f'blend.t{int(nbm_init_hour):02}z.core.{nbm_init_filen}{nbm_init_filen}f{int(nbm_qmd_forecasthour):03}.{domain}.{elements}_Det_subset_exp.grib2'
      perc_url_exp = nbm_url_base_exp+"core/"+perc_file
      search_string = build_search_string(int(nbm_qmd_forecasthour), field, SnowQPFPctl_Duration, elements)
      titles.append(f"NBM v5.0 - NBM v4.3 Deterministic {elements.replace('-Pctl','')}")
    elif elements in ["QPF-Pctl", "Snow-Pctl"] and field != "Det":
      perc_file = f'blend.t{int(nbm_init_hour):02}z.qmd.f{int(nbm_qmd_forecasthour):03}.{domain}.grib2'
      perc_url = nbm_url_base+"qmd/"+perc_file
      perc_file_subset = f'blend.t{int(nbm_init_hour):02}z.qmd.{nbm_init_filen}{nbm_init_filen}f{int(nbm_qmd_forecasthour):03}.{domain}.{elements}_{field.replace("%","")}_subset.grib2'
      perc_file_exp = f'blend.t{int(nbm_init_hour):02}z.qmd.f{int(nbm_qmd_forecasthour):03}.{domain}.grib2'
      perc_url_exp = nbm_url_base_exp+"qmd/"+perc_file
      perc_file_subset_exp = f'blend.t{int(nbm_init_hour):02}z.qmd.{nbm_init_filen}{nbm_init_filen}f{int(nbm_qmd_forecasthour):03}.{domain}.{elements}_{field.replace("%","")}_subset_exp.grib2'
      search_string = build_search_string(int(nbm_qmd_forecasthour), field, SnowQPFPctl_Duration, elements)
      titles.append(f"NBM v5.0 - NBM v4.3 {SnowQPFPctl_Duration} {elements.replace('-Pctl','')} {field.replace('%', 'th')} Percentile")
    elif elements in ["Wind-Pctl", "Gust-Pctl"] and field != "Det":
      perc_file = f'blend.t{int(nbm_init_hour):02}z.qmd.f{int(nbm_qmd_forecasthour):03}.{domain}.grib2'
      perc_url = nbm_url_base+"qmd/"+perc_file
      perc_file_subset = f'blend.t{int(nbm_init_hour):02}z.qmd.{nbm_init_filen}{nbm_init_filen}f{int(nbm_qmd_forecasthour):03}.{domain}.{elements}_{field.replace("%","")}_subset.grib2'
      perc_file_exp = f'blend.t{int(nbm_init_hour):02}z.qmd.f{int(nbm_qmd_forecasthour):03}.{domain}.grib2'
      perc_url_exp = nbm_url_base_exp+"qmd/"+perc_file
      perc_file_subset_exp = f'blend.t{int(nbm_init_hour):02}z.qmd.{nbm_init_filen}{nbm_init_filen}f{int(nbm_qmd_forecasthour):03}.{domain}.{elements}_{field.replace("%","")}_subset_exp.grib2'
      search_string = build_search_string(int(nbm_qmd_forecasthour), field, SnowQPFPctl_Duration, elements)
      titles.append(f"NBM v5.0 - NBM v4.3 {elements.replace('-Pctl','')} {field.replace('%', 'th')} Percentile")

    print(f"Field is: {field}")
    print(f"NBM operational file is: {perc_file}")
    print(f"NBM operational subset is: {perc_file_subset}")
    print(f"NBM operational url is: {perc_url}")
    print(f"NBM experimental file is: {perc_file_exp}")
    print(f"NBM experimental subset is: {perc_file_subset_exp}")
    print(f"NBM experimental url is: {perc_url_exp}")
    print(f"Searching for: {search_string}")
    if not os.path.exists(f'./nbm/{perc_file_subset}'):
      print(f"Downloading {perc_file_subset}")
    #downloading operational data
      if field != "Det":
        download_subset_dev(perc_url, perc_file, perc_file_subset, search_string)
      else:
        download_subset_dev(perc_url, perc_file, perc_file_subset, search_string, percentile=False)
    if not os.path.exists(f'./nbm/{perc_file_subset_exp}'):
      print(f"Downloading {perc_file_subset_exp}")
      if field != "Det":
        # downloading experimental data
        download_subset_dev(perc_url_exp, perc_file_exp, perc_file_subset_exp, search_string)
      else:
        #downloading experimental data
        download_subset_dev(perc_url_exp, perc_file_exp, perc_file_subset_exp, search_string, percentile=False)
    # appending filenames for later
    oper_files[field] = perc_file_subset
    exp_files[field] = perc_file_subset_exp
  #print(oper_files)
  #print(exp_files)
  #Differencing and plotting
  diff_files = []
  for (pctl, op_file), (pctl_exp, exp_file) in zip(oper_files.items(), exp_files.items()):
      # Subtract operational from experimental and save to temp
      g1 = pygrib.open(f"./nbm/{op_file}")[1]
      expgrbs = pygrib.open(f"./nbm/{exp_file}")
      #for g in expgrbs:
      #  print(g)
      g2 = expgrbs[1]
      #print(g2)

      diff = g2.values - g1.values
      #print(f"Diff is: {diff[diff>0]}")
      # Save diff to temporary numpy array for plotting (or just pass values directly)
      # For simplicity we'll just modify the plot function to handle numpy arrays
      diff_files.append(diff)
  # Modified plot function would need to accept raw data too
  # setting colormaps
  nbmfiles = [filename for field, filename in oper_files.items()]
  plot_six_panel_grib2(elements, nbmfiles, titles, Map_Extent, crs, cmap, cbar, Wind_Units, diffdata=diff_files, difference_mode=True)
else:
  nbm_init_filen = nbm_init.strftime('%Y%m%d') + "_" + nbm_init.strftime('%H')
  #nbm_init_filen_core = core_init.strftime('%Y%m%d') + "_" + core_init.strftime('%H')
  nbm_init_filen_core = nbm_init_filen

  if Model == "NBM v4.3":
    nbm_url_base = "https://noaa-nbm-grib2-pds.s3.amazonaws.com/blend."+nbm_init.strftime('%Y%m%d') \
                +"/"+nbm_init.strftime('%H')+"/"
    nbm_url_base_core = "https://noaa-nbm-grib2-pds.s3.amazonaws.com/blend."+core_init.strftime('%Y%m%d') \
                +"/"+core_init.strftime('%H')+"/"
  else:
    nbm_url_base = "https://noaa-nbm-para-pds.s3.amazonaws.com/blend."+nbm_init.strftime('%Y%m%d') \
                +"/"+nbm_init.strftime('%H')+"/"
    nbm_url_base_core = "https://noaa-nbm-para-pds.s3.amazonaws.com/blend."+core_init.strftime('%Y%m%d') \
                +"/"+core_init.strftime('%H')+"/"
  #print(nbm_url_base)
  #print(nbm_url_base_core)
  # downloading our data
  oper_files = {}
  titles = []
  print(f"Using domain of {domain} to pull data...")
  for field in [Pctl_UL, Pctl_LL, Pctl_UM, Pctl_LM, Pctl_UR, Pctl_LR]:
    if elements == "QPF-Pctl" and field == "Det":
      perc_file = f'blend.t{int(nbm_init_hour):02}z.qmd.f{int(nbm_qmd_forecasthour):03}.{domain}.grib2'
      if Model == "NBM v5.0":
        perc_file_subset = f'blend.t{int(nbm_init_hour):02}z.qmd.{nbm_init_filen}{nbm_init_filen}f{int(nbm_qmd_forecasthour):03}.{domain}.{elements}_Det_subset_exp.grib2'
      else:
        perc_file_subset = f'blend.t{int(nbm_init_hour):02}z.qmd.{nbm_init_filen}{nbm_init_filen}f{int(nbm_qmd_forecasthour):03}.{domain}.{elements}_Det_subset.grib2'
      perc_url = nbm_url_base+"qmd/"+perc_file
      search_string = build_search_string(int(nbm_qmd_forecasthour), field, SnowQPFPctl_Duration, elements)
      titles.append(f"{Model} {SnowQPFPctl_Duration} Deterministic")
    elif elements == "QPF-Pctl" and field != "Det":
      perc_file = f'blend.t{int(nbm_init_hour):02}z.qmd.f{int(nbm_qmd_forecasthour):03}.{domain}.grib2'
      perc_url = nbm_url_base+"qmd/"+perc_file
      if Model == "NBM v5.0":
        perc_file_subset = f'blend.t{int(nbm_init_hour):02}z.qmd.{nbm_init_filen}{nbm_init_filen}f{int(nbm_qmd_forecasthour):03}.{domain}.{elements}_{field.replace("%","")}_subset_exp.grib2'
      else:
        perc_file_subset = f'blend.t{int(nbm_init_hour):02}z.qmd.{nbm_init_filen}{nbm_init_filen}f{int(nbm_qmd_forecasthour):03}.{domain}.{elements}_{field.replace("%","")}_subset.grib2'
      search_string = build_search_string(int(nbm_qmd_forecasthour), field, SnowQPFPctl_Duration, elements)
      titles.append(f"{Model} {SnowQPFPctl_Duration} {elements.replace('-Pctl','')} {field.replace('%','th')} Percentile")
    elif elements == "Wind-Pctl" and field != "Det":
      if domain == "ak" and Model == "NBM v4.3":
        print(f"Wind percentiles don't exist for {Model}. Try NBM v5.0")
        raise SystemExit
      else:
        perc_file = f'blend.t{int(nbm_init_hour):02}z.qmd.f{int(nbm_qmd_forecasthour):03}.{domain}.grib2'
        perc_url = nbm_url_base+"qmd/"+perc_file
        if Model == "NBM v5.0":
          perc_file_subset = f'blend.t{int(nbm_init_hour):02}z.qmd.{nbm_init_filen}{nbm_init_filen}f{int(nbm_qmd_forecasthour):03}.{domain}.{elements}_{field.replace("%","")}_subset_exp.grib2'
        else:
          perc_file_subset = f'blend.t{int(nbm_init_hour):02}z.qmd.{nbm_init_filen}{nbm_init_filen}f{int(nbm_qmd_forecasthour):03}.{domain}.{elements}_{field.replace("%","")}_subset_exp.grib2'
        search_string = build_search_string(int(nbm_qmd_forecasthour), field, SnowQPFPctl_Duration, elements)
        titles.append(f"{Model} {elements.replace('-Pctl','')} {field.replace('%','th')} Percentile")
    elif elements == "Wind-Pctl" and field == "Det":
      if domain == "ak" and Model == "NBM v4.3":
        print(f"Wind percentiles don't exist for {Model}. Try NBM v5.0")
        raise SystemExit
      else:
        perc_file = f'blend.t{int(nbm_init_hour):02}z.core.f{int(nbm_qmd_forecasthour):03}.{domain}.grib2'
        perc_url = nbm_url_base+"core/"+perc_file
        if Model == "NBM v5.0":
          perc_file_subset = f'blend.t{int(nbm_init_hour):02}z.core.{nbm_init_filen}{nbm_init_filen}f{int(nbm_qmd_forecasthour):03}.{domain}.{elements}_{field.replace("%","")}_subset_exp.grib2'
        else:
          perc_file_subset = f'blend.t{int(nbm_init_hour):02}z.core.{nbm_init_filen}{nbm_init_filen}f{int(nbm_qmd_forecasthour):03}.{domain}.{elements}_{field.replace("%","")}_subset.grib2'
        search_string = build_search_string(int(nbm_qmd_forecasthour), field, SnowQPFPctl_Duration, elements)
        titles.append(f"{Model} {elements.replace('-Pctl','')} {field.replace('%','th')}")



    #print(f"Field is: {field}")
    #print(perc_file)
    #print(perc_file_subset)
    #print(perc_url)
    #print(search_string)
    #print(f"Titles are: {titles}")
    if not os.path.exists(f'./nbm/{perc_file_subset}'):
      print(f"Downloading {perc_file_subset}")
      if field != "Det":
        download_subset_dev(perc_url, perc_file, perc_file_subset, search_string)
      else:
        download_subset_dev(perc_url, perc_file, perc_file_subset, search_string, percentile=False)
    # Appending our filenames
    oper_files[field] = perc_file_subset
  # plotting our data
  nbmfiles = [filename for field, filename in oper_files.items()]
  plot_six_panel_grib2(elements, nbmfiles, titles, Map_Extent, crs, cmap, cbar, Wind_Units)





/tmp/ipython-input-3102202245.py:25: UserWarning: Overwriting the cmap 'nws.pcp' that was already in the registry.
  mpl.colormaps.register(cmap=cm, force=True)
/tmp/ipython-input-3102202245.py:26: UserWarning: Overwriting the cmap 'nws.pcp_r' that was already in the registry.
  mpl.colormaps.register(cmap=cm.reversed(), force=True)
/tmp/ipython-input-3102202245.py:25: UserWarning: Overwriting the cmap 'nws.pcp2' that was already in the registry.
  mpl.colormaps.register(cmap=cm, force=True)
/tmp/ipython-input-3102202245.py:26: UserWarning: Overwriting the cmap 'nws.pcp2_r' that was already in the registry.
  mpl.colormaps.register(cmap=cm.reversed(), force=True)
/tmp/ipython-input-3102202245.py:25: UserWarning: Overwriting the cmap 'nws.pcp_snow' that was already in the registry.
  mpl.colormaps.register(cmap=cm, force=True)
/tmp/ipython-input-3102202245.py:26: UserWarning: Overwriting the cmap 'nws.pcp_snow_r' that was already in the registry.
  mpl.colormaps.register(cmap=cm.reverse

Using Map_Zoom list logic...
Domain is: ak
Using domain of ak to pull data...
Field is: 10%
NBM operational file is: blend.t00z.qmd.f036.ak.grib2
NBM operational subset is: blend.t00z.qmd.20250731_0020250731_00f036.ak.QPF-Pctl_10_subset.grib2
NBM operational url is: https://noaa-nbm-grib2-pds.s3.amazonaws.com/blend.20250731/00/qmd/blend.t00z.qmd.f036.ak.grib2
NBM experimental file is: blend.t00z.qmd.f036.ak.grib2
NBM experimental subset is: blend.t00z.qmd.20250731_0020250731_00f036.ak.QPF-Pctl_10_subset_exp.grib2
NBM experimental url is: https://noaa-nbm-para-pds.s3.amazonaws.com/blend.20250731/00/qmd/blend.t00z.qmd.f036.ak.grib2
Searching for: :APCP:surface:12-36 hour acc fcst:10% level
Field is: 50%
NBM operational file is: blend.t00z.qmd.f036.ak.grib2
NBM operational subset is: blend.t00z.qmd.20250731_0020250731_00f036.ak.QPF-Pctl_50_subset.grib2
NBM operational url is: https://noaa-nbm-grib2-pds.s3.amazonaws.com/blend.20250731/00/qmd/blend.t00z.qmd.f036.ak.grib2
NBM experimental fi

# **Zip up image files for download**
This will zip all .png files in the folder to the left for download.

In [7]:
!zip -r NBM_Data.zip *.png

  adding: NBMv4.3_QPF-Pctl_2025-07-31_2025-08-01_12.png (deflated 3%)
  adding: NBMv5.0_-NBM_QPF-Pctl_2025-07-31_2025-08-01_12.png (deflated 2%)
  adding: NBMv5.0_QPF-Pctl_2025-07-31_2025-08-01_12.png (deflated 3%)


# **Remove Images in Folder to the left**
If you're saving a lot of images and want to clear the folder to the left, run the section below.

In [8]:
!rm *.png
!rm *.zip

**Change Log**

**V1.23** - Added option to display a list of percentiles for a Lat/Lon point. <br>
**V1.22** - Added Marine Zones to the Great Lakes map zooms. <br>
**V1.21** - Fixed issue with Wind Percentiles and added Great Lakes Map Zooms (thanks Dan T!).  <br>
**V1.2** - Added Wind Percentiles.  <br>
**V1.1** - Added QPF Percentiles.  <br>
**V1.02** - Fixed issue with ecCodes 2.39.0 and Herbie <br>
**V1.01** - Added option to Zip files and remove when creating multiple.  <br>
**V1.0** - Original Release <br>